# 10 — Búsqueda de lugares e instituciones de salud

Equivalente al notebook `12_busqueda_lugares_personajes.ipynb` de Karen,  
adaptado al dominio de salud: busca por regex departamentos colombianos,  
ciudades principales e instituciones de salud en los tweets.

**Entrada:** `corpus_cleaned.parquet`, `salud_tweets_final.parquet`, `general_ner.parquet`  
**Salida:** `resultados_lugares_salud.parquet`, `resultados_instituciones_salud.parquet`, Excel

In [ ]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios')

print('[CONFIG] OK')


In [ ]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# ============================================================
import re
import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Corpus completo
corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')
corpus['Fecha'] = pd.to_datetime(corpus['Fecha'], errors='coerce')

# Subcorpus de salud
tweets_salud = pd.read_parquet(DATA_PROCESSED / 'salud_tweets_final.parquet')

# Merge para obtener texto
df_salud = tweets_salud.merge(
    corpus[['id_doc', 'Texto_limpio', 'Fecha', 'Author_Normalized', 'Entidad']],
    on='id_doc', how='left'
)
df_salud['Anio'] = df_salud['Fecha'].dt.year

print(f'Tweets de salud: {len(df_salud):,}')
df_salud.head(2)


In [ ]:
# ============================================================
# CELL 2 — DICCIONARIOS DE TÉRMINOS
# (equivalente a los Excel externos de Karen)
# ============================================================

# --- Departamentos de Colombia ---
departamentos = {
    'Amazonas'        : ['Amazonas'],
    'Antioquia'       : ['Antioquia', 'Medellin', 'Medellín'],
    'Arauca'          : ['Arauca'],
    'Atlántico'       : ['Atlantico', 'Barranquilla'],
    'Bolívar'         : ['Bolivar', 'Cartagena'],
    'Boyacá'          : ['Boyaca', 'Tunja'],
    'Caldas'          : ['Caldas', 'Manizales'],
    'Caquetá'         : ['Caqueta', 'Florencia'],
    'Casanare'        : ['Casanare', 'Yopal'],
    'Cauca'           : ['Cauca', 'Popayan'],
    'Cesar'           : ['Cesar', 'Valledupar'],
    'Chocó'           : ['Choco', 'Quibdo'],
    'Córdoba'         : ['Cordoba', 'Monteria'],
    'Cundinamarca'    : ['Cundinamarca', 'Bogota', 'Bogotá'],
    'Guainía'         : ['Guainia'],
    'Guaviare'        : ['Guaviare'],
    'Huila'           : ['Huila', 'Neiva'],
    'La Guajira'      : ['Guajira', 'Riohacha'],
    'Magdalena'       : ['Magdalena', 'Santa Marta'],
    'Meta'            : ['Meta', 'Villavicencio'],
    'Nariño'          : ['Narino', 'Pasto'],
    'Norte de Santander': ['Norte de Santander', 'Cucuta', 'Cúcuta'],
    'Putumayo'        : ['Putumayo', 'Mocoa'],
    'Quindío'         : ['Quindio', 'Armenia'],
    'Risaralda'       : ['Risaralda', 'Pereira'],
    'San Andrés'      : ['San Andres'],
    'Santander'       : ['Santander', 'Bucaramanga'],
    'Sucre'           : ['Sucre', 'Sincelejo'],
    'Tolima'          : ['Tolima', 'Ibague', 'Ibagué'],
    'Valle del Cauca' : ['Valle del Cauca', 'Cali'],
    'Vaupés'          : ['Vaupes', 'Mitu'],
    'Vichada'         : ['Vichada'],
}

# --- Instituciones de salud colombianas ---
instituciones = {
    'Ministerio de Salud'              : ['Ministerio de Salud', 'MinSalud', 'Minsalud'],
    'INS'                              : ['INS', 'Instituto Nacional de Salud'],
    'INVIMA'                           : ['INVIMA', 'Invima'],
    'Secretaría de Salud'              : ['Secretaria de Salud', 'Secretaría de Salud', 'SecSalud'],
    'OMS/OPS'                          : ['OMS', 'OPS', 'PAHO', 'WHO'],
    'UNICEF'                           : ['UNICEF', 'Unicef'],
    'Cruz Roja'                        : ['Cruz Roja'],
    'EPS'                              : ['EPS'],
    'IPS'                              : ['IPS'],
    'ICBF'                             : ['ICBF'],
    'Médicos Sin Fronteras'            : ['Medicos Sin Fronteras', 'MSF'],
    'Sura'                             : ['Sura', 'EPS Sura'],
    'Colsanitas'                       : ['Colsanitas'],
    'Nueva EPS'                        : ['Nueva EPS'],
    'Coomeva'                          : ['Coomeva'],
    'Gobierno Colombia'                : ['Gobierno Colombia', 'Gobierno Nacional'],
    'Presidencia'                      : ['Presidencia', 'Presidente'],
    'Hospital'                         : ['Hospital', 'Clínica', 'Clinica', 'UCI', 'UTI'],
    'SGSSS'                            : ['SGSSS', 'Sistema General de Seguridad Social'],
}

print(f'Departamentos definidos   : {len(departamentos)}')
print(f'Instituciones definidas   : {len(instituciones)}')


In [ ]:
# ============================================================
# CELL 3 — FUNCIÓN DE NORMALIZACIÓN Y BÚSQUEDA
# (Replica de Karen — misma lógica, adaptada a tweets)
# ============================================================

def normalizar_texto(texto: str) -> str:
    """Minúsculas, sin tildes, sin puntos."""
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = texto.replace('.', '')
    return texto


def buscar_terminos(df_tweets: pd.DataFrame,
                    diccionario: dict,
                    col_salida: str) -> pd.DataFrame:
    """
    Busca términos (diccionario: {nombre_canonico: [variantes]})
    en el texto de los tweets.

    Retorna DataFrame con col_salida, id_doc, autor, fecha, variante hallada, texto.
    """
    textos_norm = df_tweets['Texto_limpio'].fillna('').map(normalizar_texto).to_numpy()
    id_docs     = df_tweets['id_doc'].to_numpy()
    autores     = df_tweets['Author_Normalized'].to_numpy()
    fechas      = df_tweets['Fecha'].to_numpy()
    subcats     = df_tweets['categoria_detectada'].to_numpy()
    textos_orig = df_tweets['Texto_limpio'].fillna('').to_numpy()

    resultados = []

    for nombre_canon, variantes in diccionario.items():
        variantes_norm = [normalizar_texto(v) for v in variantes if v.strip()]
        patron = re.compile(
            r'\b(' + '|'.join(re.escape(v) for v in variantes_norm) + r')\b'
        )

        for idx in range(len(textos_norm)):
            m = patron.search(textos_norm[idx])
            if m:
                resultados.append({
                    col_salida         : nombre_canon,
                    'Variante_hallada' : m.group(0),
                    'id_doc'           : id_docs[idx],
                    'autor'            : autores[idx],
                    'Fecha'            : fechas[idx],
                    'categoria_detectada': subcats[idx],
                    'Texto_limpio'     : textos_orig[idx],
                })

    df_res = pd.DataFrame(resultados)
    if not df_res.empty:
        df_res = (
            df_res
            .drop_duplicates(subset=[col_salida, 'id_doc'])
            .reset_index(drop=True)
        )
    return df_res


In [ ]:
# ============================================================
# CELL 4 — BÚSQUEDA EN CORPUS COMPLETO
# (lugares e instituciones en todos los tweets)
# ============================================================
print('Buscando departamentos en corpus completo...')
df_res_depto = buscar_terminos(
    df_tweets=df_salud,
    diccionario=departamentos,
    col_salida='Departamento'
)
print(f'  Departamentos encontrados : {len(df_res_depto):,} menciones')

print('Buscando instituciones en corpus de salud...')
df_res_inst = buscar_terminos(
    df_tweets=df_salud,
    diccionario=instituciones,
    col_salida='Institucion'
)
print(f'  Instituciones encontradas : {len(df_res_inst):,} menciones')


In [ ]:
# ============================================================
# CELL 5 — TOP DEPARTAMENTOS
# ============================================================
freq_depto = (
    df_res_depto.groupby('Departamento').size()
    .reset_index(name='Frecuencia')
    .sort_values('Frecuencia', ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 7))
freq_depto_top = freq_depto.head(20)
freq_depto_top_sorted = freq_depto_top.sort_values('Frecuencia', ascending=True)
ax.barh(freq_depto_top_sorted['Departamento'], freq_depto_top_sorted['Frecuencia'], color='teal')
ax.set_title('Top 20 departamentos mencionados en tweets de salud')
ax.set_xlabel('Número de menciones')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_top_departamentos_salud.png', dpi=300)
plt.show()
print('[GUARDADO] fig_top_departamentos_salud.png')
print(freq_depto.head(20).to_string(index=False))


In [ ]:
# ============================================================
# CELL 6 — TOP INSTITUCIONES
# ============================================================
freq_inst = (
    df_res_inst.groupby('Institucion').size()
    .reset_index(name='Frecuencia')
    .sort_values('Frecuencia', ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 6))
freq_inst_s = freq_inst.sort_values('Frecuencia', ascending=True)
ax.barh(freq_inst_s['Institucion'], freq_inst_s['Frecuencia'], color='darkorange')
ax.set_title('Instituciones de salud mencionadas en tweets')
ax.set_xlabel('Número de menciones')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_top_instituciones_salud.png', dpi=300)
plt.show()
print('[GUARDADO] fig_top_instituciones_salud.png')
print(freq_inst.to_string(index=False))


In [ ]:
# ============================================================
# CELL 7 — EVOLUCIÓN TEMPORAL DE LUGARES E INSTITUCIONES
# ============================================================
df_res_depto['Anio'] = pd.to_datetime(df_res_depto['Fecha']).dt.year

evol_depto = (
    df_res_depto.groupby(['Anio', 'Departamento']).size()
    .reset_index(name='n')
)

# Top 8 departamentos para la visualización
top8_deptos = freq_depto.head(8)['Departamento'].tolist()
evol_depto_top = evol_depto[evol_depto['Departamento'].isin(top8_deptos)]

fig, ax = plt.subplots(figsize=(10, 5))
for depto in top8_deptos:
    sub = evol_depto_top[evol_depto_top['Departamento'] == depto]
    ax.plot(sub['Anio'].astype(str), sub['n'], marker='o', label=depto)

ax.set_title('Evolución anual de menciones geográficas — top 8 departamentos')
ax.set_ylabel('Tweets')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_evolucion_departamentos.png', dpi=300)
plt.show()
print('[GUARDADO] fig_evolucion_departamentos.png')


In [ ]:
# ============================================================
# CELL 8 — NER: ANÁLISIS DE ENTIDADES LOC DESDE salud_ner
# (complementa la búsqueda por regex con las entidades ya extraídas)
# ============================================================
salud_ner = pd.read_parquet(DATA_PROCESSED / 'salud_ner.parquet')

# Aplanar entidades LOC
loc_rows = []
for _, row in salud_ner.iterrows():
    for ent in row['entidades']:
        if len(ent) == 2 and ent[0] == 'LOC':
            loc_rows.append({
                'id_doc'   : row['id_doc'],
                'Anio'     : row['Anio'],
                'Entidad'  : str(ent[1]).strip(),
            })

df_loc_ner = pd.DataFrame(loc_rows)

freq_loc = (
    df_loc_ner.groupby('Entidad').size()
    .reset_index(name='Frecuencia')
    .sort_values('Frecuencia', ascending=False)
)

print(f'Entidades LOC (NER) en subcorpus de salud: {len(df_loc_ner):,} menciones')
print(f'Entidades únicas: {freq_loc["Entidad"].nunique():,}')
print()
print('Top 20 lugares (NER):')
print(freq_loc.head(20).to_string(index=False))


In [ ]:
# ============================================================
# CELL 9 — GUARDAR RESULTADOS
# ============================================================

# Asegurar tipos string para parquet
for col in ['Departamento', 'Texto_limpio', 'Variante_hallada']:
    if col in df_res_depto.columns:
        df_res_depto[col] = df_res_depto[col].astype(str)

for col in ['Institucion', 'Texto_limpio', 'Variante_hallada']:
    if col in df_res_inst.columns:
        df_res_inst[col] = df_res_inst[col].astype(str)

df_res_depto.to_parquet(DATA_PROCESSED / 'resultados_lugares_salud.parquet', index=False)
df_res_inst.to_parquet(DATA_PROCESSED / 'resultados_instituciones_salud.parquet', index=False)

ruta_excel = DATA_PROCESSED / 'busqueda_lugares_instituciones_salud.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    freq_depto.to_excel(writer, sheet_name='Frecuencia_departamentos', index=False)
    freq_inst.to_excel(writer,  sheet_name='Frecuencia_instituciones', index=False)
    df_res_depto.drop(columns=['Texto_limpio'], errors='ignore').to_excel(
        writer, sheet_name='Menciones_departamentos', index=False
    )
    df_res_inst.drop(columns=['Texto_limpio'], errors='ignore').to_excel(
        writer, sheet_name='Menciones_instituciones', index=False
    )
    freq_loc.head(100).to_excel(writer, sheet_name='LOC_NER_top100', index=False)

print('[GUARDADO] resultados_lugares_salud.parquet')
print('[GUARDADO] resultados_instituciones_salud.parquet')
print('[GUARDADO] busqueda_lugares_instituciones_salud.xlsx')
print()
print('Notebook 10 completado.')
print('Siguiente -> 11_descriptivos_avanzados_salud.ipynb')
